<a href="https://colab.research.google.com/github/Yashas7206988696/Transfer-Learning/blob/main/finetuned_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [84]:
import torch
from transformers import AutoTokenizer as at, AutoModelForSequenceClassification as ams

In [85]:
# load the pre-trained model
tokenizer = at.from_pretrained("distilbert-base-uncased")
model = ams.from_pretrained("distilbert-base-uncased")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [86]:
#define hyperparametres
learning_rate = 5
epochs = 5

In [87]:
#load the training data
xtrain = torch.load("xtrain.pt")

In [88]:
import torch

xtrain = [
    # Positive
    "I loved this movie, it was fantastic and uplifting!",
    "Excellent product, works just as described.",
    "What a great experience—I would recommend it to everyone.",
    "The service was outstanding, truly a delight.",
    "Brilliant job! I am very satisfied.",
    # Negative
    "I hated this movie, it was boring and too long.",
    "Terrible product, broke after one use.",
    "The experience was disappointing and frustrating.",
    "Customer service was rude and unhelpful.",
    "Awful, I will never buy this again."
]

# Save text samples to file for later use
torch.save(xtrain, "xtrain.pt")

print("Saved text training samples to xtrain.pt")


Saved text training samples to xtrain.pt


In [89]:
#create  a optimizer
optim = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [90]:
#train the model
encoding = tokenizer(xtrain, padding=True, truncation=True, return_tensors="pt")

for epoch in range(epochs):
    for i in range(len(xtrain)):
        batch = {
            "input_ids": encoding["input_ids"][i].unsqueeze(0),
            "attention_mask": encoding["attention_mask"][i].unsqueeze(0),
            "labels": labels[i].unsqueeze(0)
        }
        outputs = model(**batch)
        loss = outputs.loss

        # Backpropagation
        optim.zero_grad()
        loss.backward()
        optim.step()


#saving the fine tuned model
torch.save(model.state_dict(), "fine-tuned_model.pt")


In [92]:
# Load the fine-tuned model
# Instantiate the model architecture
model = ams.from_pretrained("distilbert-base-uncased")
# Load the saved state dictionary
model.load_state_dict(torch.load("fine-tuned_model.pt"))
model.eval() # Set the model to evaluation mode

# Classify a text sequence
text_sequence = "This is a positive review."
# Tokenize the input text
inputs = tokenizer(text_sequence, return_tensors="pt")

# Perform inference
with torch.no_grad(): # Disable gradient calculation for inference
    outputs = model(**inputs)

# Get the predicted label
# The output logits will be probabilities for each class
predicted_label = torch.argmax(outputs.logits, dim=-1)

# Print the predicted label
print(f"Text: '{text_sequence}'")
print(f"Predicted label (0 for negative, 1 for positive based on dummy labels): {predicted_label.item()}")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Text: 'This is a positive review.'
Predicted label (0 for negative, 1 for positive based on dummy labels): 0
